In [1]:
!pip install tabulate
# install package, '!' to run shell commands in notebook

In [2]:
#install necessary packages
import pandas as pd 
import numpy as np
from tabulate import tabulate

#function to format a dataframe into nicely formatted tables
#headers uses column names as table headers
def print_table(df):
  return print(tabulate(df,headers = 'keys', tablefmt='fancy_grid'))
    
# Load the dataset; sep specifies the file is tab-separated
df = pd.read_csv("expvalues.txt", sep="\t")

# Display the first few rows of the dataset
print_table(df.head())

╒════╤══════════════╤════════════╤════════════╤════════════╤══════════════╤══════════════╤══════════════╕
│    │ Unnamed: 0   │   Control1 │   Control2 │   Control3 │   Treatment1 │   Treatment2 │   Treatment3 │
╞════╪══════════════╪════════════╪════════════╪════════════╪══════════════╪══════════════╪══════════════╡
│  0 │ 244901_at    │   229.986  │   353.599  │   178.492  │     112.908  │     152.918  │     320.832  │
├────┼──────────────┼────────────┼────────────┼────────────┼──────────────┼──────────────┼──────────────┤
│  1 │ 244902_at    │   171.15   │    84.4509 │    41.372  │     170.173  │     134.408  │     193.526  │
├────┼──────────────┼────────────┼────────────┼────────────┼──────────────┼──────────────┼──────────────┤
│  2 │ 244903_at    │   314.978  │   373.312  │    52.9087 │     196.303  │     237.515  │     253.378  │
├────┼──────────────┼────────────┼────────────┼────────────┼──────────────┼──────────────┼──────────────┤
│  3 │ 244904_at    │    24.0437 │   175.316  

In [3]:
# pivot dataframe; transform it from wide to long format
# id_vars, list comprehension; store columns that do not start with control or treatment in list
df2 = df.melt(
  id_vars=[col for col in df.columns if not col.startswith(('Control', 'Treatment'))],
  value_vars=[col for col in df.columns if col.startswith(('Control', 'Treatment'))], #stores columns that start with treatment or control that we want to convert into rows
  var_name='expname', value_name='expvalue' # create new column, 'expname' to store experiment names; 'expvalue; to store expression values

)
#remove digits at the end of the experiment names
df2['expname'] = df2['expname'].str.replace(r'\d+', '', regex=True)

In [4]:
print(df2)
#display the transformed dataframe

     Unnamed: 0    expname    expvalue
0     244901_at    Control  229.985654
1     244902_at    Control  171.149796
2     244903_at    Control  314.977684
3     244904_at    Control   24.043665
4     244905_at    Control   15.869235
...         ...        ...         ...
1573  245159_at  Treatment  126.136578
1574  245160_at  Treatment   68.410565
1575  245161_at  Treatment  242.856839
1576  245162_at  Treatment   10.144342
1577  245163_at  Treatment  186.989457

[1578 rows x 3 columns]


In [5]:
df2 = df2.rename(columns = {'Unnamed: 0':'probename'})
#rename unnamed column to probename

print(df2)
#print to check name change

      probename    expname    expvalue
0     244901_at    Control  229.985654
1     244902_at    Control  171.149796
2     244903_at    Control  314.977684
3     244904_at    Control   24.043665
4     244905_at    Control   15.869235
...         ...        ...         ...
1573  245159_at  Treatment  126.136578
1574  245160_at  Treatment   68.410565
1575  245161_at  Treatment  242.856839
1576  245162_at  Treatment   10.144342
1577  245163_at  Treatment  186.989457

[1578 rows x 3 columns]


In [6]:
experiment = df2[['expname']].drop_duplicates().reset_index(drop=True) #remove duplicate rows from the expname column
#reset_index(drop=True) drop prior index and assign new index
experiment = experiment.reset_index(names="expid") # store reset index in new column called expid
print(experiment)

   expid    expname
0      0    Control
1      1  Treatment


In [7]:
probes = df2[["probename"]].drop_duplicates() #remove duplicate rows from the probename column 
probes = probes.reset_index(names = "probeid") #reset dataframe index and store in a column called probeid

print(probes) #display df to check

     probeid  probename
0          0  244901_at
1          1  244902_at
2          2  244903_at
3          3  244904_at
4          4  244905_at
..       ...        ...
258      258  245159_at
259      259  245160_at
260      260  245161_at
261      261  245162_at
262      262  245163_at

[263 rows x 2 columns]


In [8]:
de1 = df2.join(experiment.set_index("expname"), on="expname") 
#join to add expid column from the experiment table to original dataframe (df2_
de1 = de1.join(probes.set_index("probename"), on="probename") 
#join to add probeid column from the probes table to original dataframe (df2)

#create new column called dataid, with reset index
df_data = de1.reset_index(names = "dataid")
#Select only these columns for the Data table
Data = df_data[["dataid","expid","probeid","expvalue"]]
print(Data)

#print 
print(Data.dtypes)

      dataid  expid  probeid    expvalue
0          0      0        0  229.985654
1          1      0        1  171.149796
2          2      0        2  314.977684
3          3      0        3   24.043665
4          4      0        4   15.869235
...      ...    ...      ...         ...
1573    1573      1      258  126.136578
1574    1574      1      259   68.410565
1575    1575      1      260  242.856839
1576    1576      1      261   10.144342
1577    1577      1      262  186.989457

[1578 rows x 4 columns]
dataid        int64
expid         int64
probeid       int64
expvalue    float64
dtype: object


In [9]:
?pd.DataFrame.join

Signature:
pd.DataFrame.join(
    self,
    other: 'DataFrame | Series | Iterable[DataFrame | Series]',
    on: 'IndexLabel | None' = None,
    how: 'MergeHow' = 'left',
    lsuffix: 'str' = '',
    rsuffix: 'str' = '',
    sort: 'bool' = False,
    validate: 'JoinValidate | None' = None,
) -> 'DataFrame'
Docstring:
Join columns of another DataFrame.

Join columns with `other` DataFrame either on index or on a key
column. Efficiently join multiple DataFrame objects by index at once by
passing a list.

Parameters
----------
other : DataFrame, Series, or a list containing any combination of them
    Index should be similar to one of the columns in this one. If a
    Series is passed, its name attribute must be set, and that will be
    used as the column name in the resulting joined DataFrame.
on : str, list of str, or array-like, optional
    Column or index level name(s) in the caller to join on the index
    in `other`, otherwise joins index-on-index. If multiple
    values given, the

In [15]:
import sqlite3

#connect to sqlite database; creates db if it doesn't exist
con = sqlite3.connect("Kalkidan_Tadese_db.sqlite")
c = con.cursor() 
# create pointer/cursor object to interact with db

c.execute("DROP TABLE IF EXISTS Experiment")
con.commit() # explicit commit db changes

#create Experiment table if it doesn't exist; provide attributes -- two columns, data types
c.execute("""CREATE TABLE IF NOT EXISTS Experiment (
expid INTEGER PRIMARY KEY, expname TEXT NOT NULL )""")

#load values from experiment dataframe into Experiment table; if table already exists, replace w new data; w/o index
experiment.to_sql("Experiment", con, if_exists='replace', index=False)

# execute sql query to select everything from the Experiment table
c.execute("""SELECT * from Experiment""")

# Fetchall - fetches all records
# as list of tuples
recordstbl1 = c.fetchall()
print(recordstbl1)

[(0, 'Control'), (1, 'Treatment')]


In [16]:
c.execute("DROP TABLE IF EXISTS Probes")
con.commit() # explicit commit changes to db


#create Probes table if it doesn't exist; provide attributes -- two columns, respective attributes
c.execute("""CREATE TABLE IF NOT EXISTS Probes ( probeid INTEGER PRIMARY KEY, probename TEXT NOT NULL )""")

#load values from probes dataframe into Probes table; if table already exists, replace w new data; w/o index
probes.to_sql("Probes", con, if_exists='replace', index=False)

# execute sql query to select everything from the Probes table
c.execute("""select * from Probes""")

# Fetch all records
recordstbl2 = c.fetchall()
print(recordstbl2)

[(0, '244901_at'), (1, '244902_at'), (2, '244903_at'), (3, '244904_at'), (4, '244905_at'), (5, '244906_at'), (6, '244907_at'), (7, '244908_at'), (8, '244909_at'), (9, '244910_s_at'), (10, '244911_at'), (11, '244912_at'), (12, '244913_at'), (13, '244914_at'), (14, '244915_s_at'), (15, '244916_at'), (16, '244917_at'), (17, '244918_at'), (18, '244919_at'), (19, '244920_s_at'), (20, '244921_s_at'), (21, '244922_s_at'), (22, '244923_s_at'), (23, '244924_at'), (24, '244925_at'), (25, '244926_s_at'), (26, '244927_at'), (27, '244928_s_at'), (28, '244929_at'), (29, '244930_at'), (30, '244931_at'), (31, '244932_at'), (32, '244933_at'), (33, '244934_at'), (34, '244935_at'), (35, '244936_at'), (36, '244937_at'), (37, '244938_at'), (38, '244939_at'), (39, '244940_at'), (40, '244941_at'), (41, '244942_at'), (42, '244943_at'), (43, '244944_s_at'), (44, '244945_at'), (45, '244946_at'), (46, '244947_at'), (47, '244948_at'), (48, '244949_at'), (49, '244950_at'), (50, '244951_s_at'), (51, '244952_at'), (

In [17]:
# execute sql query to drop Data table if it already exists; drop to remove potential conflict w later create Data query
c.execute("DROP TABLE IF EXISTS Data")
con.commit() # explicit commit changes to db

#create Data table if it doesn't exist; provide attributes -- four columns, respective attributes
c.execute("""CREATE TABLE IF NOT EXISTS Data ( 
dataid INTEGER PRIMARY KEY AUTOINCREMENT, 
expid INTEGER, 
probeid INTEGER, 
expvalue REAL, 
FOREIGN KEY (expid) REFERENCES Experiment(expid), 
FOREIGN KEY (probeid) REFERENCES Probes(probeid))""")

#load values from Data df into Data table; if table already exists, add data; w/o index
Data.to_sql("Data", con, if_exists='append', index=False)
con.commit() #commit changes to db

c.execute("PRAGMA table_info(Data)") # query information about the Data table -- to confirm table format
print(c.fetchall()) ## Fetchall - fetches all records stored in c and prints them

[(0, 'dataid', 'INTEGER', 0, None, 1), (1, 'expid', 'INTEGER', 0, None, 0), (2, 'probeid', 'INTEGER', 0, None, 0), (3, 'expvalue', 'REAL', 0, None, 0)]


In [21]:
print(Data.head())  # Preview some rows


   dataid  expid  probeid    expvalue
0       0      0        0  229.985654
1       1      0        1  171.149796
2       2      0        2  314.977684
3       3      0        3   24.043665
4       4      0        4   15.869235
